# **实验三：基于 CNN 的手写数字识别**

使用 TensorFlow/Keras 构建卷积神经网络，在 MNIST 数据集上完成手写数字（0-9）的分类。

- **网络结构**：两层卷积（$32\rarr64$ 个 $3\times3$ 卷积核）+ 最大池化提取空间特征，两层全连接层（$3136\rarr128\rarr10$）完成分类。
- **损失函数**：交叉熵，优化器为 Adam（$lr=0.001$）。

训练 5 个 epoch 后在 10,000 张测试图片上评估准确率。

辅助模块：

- `CNN.py`：CNN 模型定义。
- `Training.py`：训练与评估函数。

In [1]:
%cd /content/Drive/MyDrive/Colab/AiLab/Lab3

/content/Drive/MyDrive/Colab/AiLab/Lab3


# **0 导入环境**

In [ ]:
import tensorflow as tf

from Sources.CNN import BuildModel
from Sources.Training import Train, Test

## **0.1 参数**

In [3]:
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.20.0
GPU available: True


# **1 数据准备**

MNIST 数据集包含 60,000 张训练图片和 10,000 张测试图片，每张为 $28\times28$ 的灰度手写数字。

TensorFlow 内置了 MNIST 数据集，可直接加载，无需联网下载。

将像素值归一化到 $[0, 1]$ 范围，并增加通道维度以匹配卷积层输入要求（$28\times28\times1$）。

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Input shape: {x_train.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 60000
Test samples: 10000
Input shape: (60000, 28, 28, 1)


# **2 模型构建**

构建卷积神经网络（CNN），结构为：
- **卷积层 1**：输入单通道灰度图，输出 32 个特征图，卷积核 $3\times3$。
- **最大池化层**：窗口 $2\times2$，步长 $2$，用于特征下采样。
- **卷积层 2**：进一步提取深层抽象特征，输出 64 个特征图。
- **全连接隐藏层**：$28\times28$ 的图像经过两次 $2\times2$ 池化后尺寸变为 $7\times7$，展平后映射到 $128$ 维特征空间。
- **输出层**：10 个节点对应数字 0-9 的分类概率。

激活函数使用 ReLU。输出层不加 Softmax，因为损失函数 `SparseCategoricalCrossentropy(from_logits=True)` 内部会自动处理。

In [8]:
model = BuildModel()
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,642 (1.61 MB)

 Trainable params: 421,642 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

# **3 运行实验**

5 轮 epoch 足以让 MNIST 达到 $98.5\%$ 以上的优秀表现。

In [6]:
Train(model, x_train, y_train, x_test, y_test, epochs=5)
Test(model, x_test, y_test)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9561 - loss: 0.1476 - val_accuracy: 0.9844 - val_loss: 0.0475
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9868 - loss: 0.0426 - val_accuracy: 0.9907 - val_loss: 0.0286
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9905 - loss: 0.0299 - val_accuracy: 0.9906 - val_loss: 0.0283
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9931 - loss: 0.0217 - val_accuracy: 0.9913 - val_loss: 0.0274
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9946 - loss: 0.0166 - val_accuracy: 0.9904 - val_loss: 0.0305

[Test] Accuracy on 10000 test images: 99.04%
